# Dia 3: EDA com SQL (DuckDB)

As consultas vivem em `sql/queries.sql` (fonte única); este notebook só as executa e desenha os gráficos.

**Regras do projeto, aplicadas em tudo:**
1. Percentagens só sobre quem respondeu (NULL nunca é "No").
2. Comparações de IA entre anos só entre utilizadores quando a pergunta mudou de base.
3. Amostra mínima por grupo (`MIN_N`) e margem de erro nas variações.
4. Resultados descritivos: associação, não causalidade.

In [ ]:
import os, sys
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")                       # trabalhar sempre a partir da raiz do projeto
sys.path.insert(0, os.getcwd())

import duckdb
import pandas as pd
import plotly.express as px

from src.eda_utils import (load_queries, setup_views, run, dumbbell,
                           delta_ci, save_fig, short_country)

pd.set_option("display.width", 160)

con = duckdb.connect()                   # base em memória
Q = load_queries("sql/queries.sql")
setup_views(con, Q)

# Parâmetros (decisões documentadas em docs/)
MIN_N = 100          # amostra mínima por país/função/tecnologia e ano
MIN_CELL = 50        # amostra mínima por célula numa mediana
MIN_CELL_TEC = 30    # idem, para país x tecnologia
ORDEM_EXP = ["0-1 anos", "2-4 anos", "5-9 anos", "10-19 anos", "20+ anos"]

con.sql("DESCRIBE survey").df()[["column_name", "column_type"]]

In [ ]:
# Verificação: 20.669 (2024) e 18.513 (2025)
con.sql("SELECT year, COUNT(*) AS n FROM survey GROUP BY year ORDER BY year").df()

## Q1. Quem usa IA?

### G1. Adoção por ano

In [ ]:
d1 = run(con, Q, "adocao_ano")
display(d1)
d1["ano"] = d1["year"].astype(str)
fig = px.bar(d1, x="ano", y="pct_usa_ia", text="pct_usa_ia",
             title="Adoção de IA entre developers empregados (2024 vs 2025)",
             labels={"pct_usa_ia": "% que usa IA", "ano": ""})
fig.update_traces(texttemplate="%{text}%", textposition="outside")
fig.update_yaxes(range=[0, 100])
fig.show(); save_fig(fig, "g01_adocao_ano")

### G2. Frequência de uso (2025, só utilizadores)

In [ ]:
d2 = run(con, Q, "frequencia_2025")
display(d2)
fig = px.bar(d2, x="pct", y="ai_use_frequency", orientation="h", text="pct",
             title="Com que frequência os developers usam IA (2025, só utilizadores)",
             labels={"pct": "% dos utilizadores", "ai_use_frequency": ""},
             category_orders={"ai_use_frequency": ["daily", "weekly", "monthly_or_less"]})
fig.update_traces(texttemplate="%{text}%")
fig.update_yaxes(autorange="reversed")
fig.show(); save_fig(fig, "g02_frequencia_2025")

### Decisão: amostra mínima por país
Olhar para os dados antes de fixar o limiar. Decisão tomada: **MIN_N = 100** (31 países, ~87% das respostas); 200 serve como teste de robustez.

In [ ]:
display(run(con, Q, "paises_n"))
display(run(con, Q, "limiares"))

### G3. Adoção por país (+ margem de erro das variações)

In [ ]:
d_pais = run(con, Q, "adocao_pais", MIN_N=MIN_N)
d_pais["country"] = d_pais["country"].map(short_country)
fig, w_pais = dumbbell(d_pais, "country", f"Adoção de IA por país (n ≥ {MIN_N} por ano)")
fig.show(); save_fig(fig, "g03_adocao_pais")
delta_ci(d_pais, "country")

### G4. Adoção por função (colunas multi-valor)
`Other (please specify):` é uma categoria-resíduo: fica fora da discussão.

In [ ]:
d_func = run(con, Q, "adocao_funcao", MIN_N=MIN_N)
d_func = d_func[d_func["funcao"] != "Other (please specify):"]
fig, w_func = dumbbell(d_func, "funcao", f"Adoção de IA por função (n ≥ {MIN_N} por ano)")
fig.show(); save_fig(fig, "g04_adocao_funcao")
delta_ci(d_func, "funcao")

### G5. Adoção por experiência

In [ ]:
d_exp = run(con, Q, "adocao_experiencia")
d_exp["ano"] = d_exp["year"].astype(str)
fig = px.bar(d_exp, x="faixa", y="pct_usa_ia", color="ano", barmode="group", text="pct_usa_ia",
             category_orders={"faixa": ORDEM_EXP}, hover_data={"n": True, "n_usa": True},
             title="Adoção de IA por anos de experiência profissional",
             labels={"pct_usa_ia": "% que usa IA", "faixa": "Experiência", "ano": "Ano"})
fig.update_traces(texttemplate="%{text}%", textposition="outside")
fig.update_yaxes(range=[0, 100])
fig.show(); save_fig(fig, "g05_adocao_experiencia")
delta_ci(d_exp, "faixa").loc[ORDEM_EXP]

## Q2 e Q3. Salário

### G6. Distribuição por ano (mediana, não média)

In [ ]:
display(run(con, Q, "salario_resumo"))
d_sal = con.sql("SELECT year::VARCHAR AS ano, comp_yearly FROM survey").df()
fig = px.box(d_sal, x="ano", y="comp_yearly", log_y=True, points=False,
             labels={"comp_yearly": "Remuneração anual (USD, escala log)", "ano": "Ano"},
             title="Distribuição da remuneração anual por ano")
fig.update_yaxes(tickvals=[5000, 10000, 20000, 50000, 100000, 200000, 500000],
                 ticktext=["5k", "10k", "20k", "50k", "100k", "200k", "500k"])
fig.show(); save_fig(fig, "g06_salario_distribuicao")

### Teste da composição: a mediana subiu dentro de cada faixa de experiência?
Se a subida dentro das faixas for menor que a global, parte do aumento vem de a amostra de 2025 ser mais sénior.

In [ ]:
d_se = run(con, Q, "salario_experiencia")
display(d_se.pivot(index="faixa", columns="year", values="n").loc[ORDEM_EXP])
w = d_se.pivot(index="faixa", columns="year", values="mediana").loc[ORDEM_EXP]
w["var_%"] = ((w[2025] / w[2024] - 1) * 100).round(1)
w

### G8. Salário de quem usa IA vs quem não usa, por experiência

In [ ]:
display(run(con, Q, "salario_ia_bruto"))          # diferença bruta, sem controlar nada
d8 = run(con, Q, "salario_ia_experiencia", MIN_CELL=MIN_CELL)
d8["grupo"] = d8["uses_ai"].map({True: "Usa IA", False: "Não usa IA"})
d8["ano"] = d8["year"].astype(str)
fig = px.bar(d8, x="faixa", y="mediana", color="grupo", barmode="group", facet_col="ano",
             category_orders={"faixa": ORDEM_EXP}, hover_data=["n"],
             title="Mediana salarial por experiência e uso de IA",
             labels={"mediana": "Mediana (USD)", "faixa": "Experiência"})
fig.show(); save_fig(fig, "g08_salario_ia_experiencia")

### G7. Salário mediano por linguagem (15 mais usadas)
Descritivo: país e experiência confundem. O modelo (Dia 4) separa estes efeitos.

In [ ]:
d7 = run(con, Q, "salario_tecnologia", MIN_N=MIN_N)
top = d7.groupby("tecnologia")["n"].sum().nlargest(15).index
d7t = d7[d7["tecnologia"].isin(top)].copy()
d7t["ano"] = d7t["year"].astype(str)
fig = px.bar(d7t.sort_values("mediana"), y="tecnologia", x="mediana", color="ano",
             barmode="group", orientation="h", hover_data=["n"],
             title="Salário mediano por linguagem (15 mais usadas)",
             labels={"mediana": "Mediana (USD)", "tecnologia": ""})
fig.update_layout(height=650)
fig.show(); save_fig(fig, "g07_salario_tecnologia")

### G9. Ranking de linguagens por país (window function)
`RANK() OVER (PARTITION BY country ...)`: o ranking recomeça em cada país. Empates dão a mesma posição (1, 1, 1, 4): muitos respondentes declaram o mesmo valor, por isso lê o mapa como patamares e não como posições exatas.

In [ ]:
d9 = run(con, Q, "ranking_tecnologia_pais", TOP_PAISES=10, TOP_TEC=10, MIN_CELL_TEC=MIN_CELL_TEC)
d9["country"] = d9["country"].map(short_country)
piv = d9.pivot(index="country", columns="tecnologia", values="ranking")
fig = px.imshow(piv, text_auto=True, color_continuous_scale="RdYlGn_r", aspect="auto",
                title="Ranking salarial das 10 linguagens mais usadas, por país (2025; 1 = mais bem paga)")
fig.show(); save_fig(fig, "g09_ranking_tecnologia_pais")
d9.head(12)

## Q4. Confiança na IA

### G10. Confiança na precisão da IA (só utilizadores, nos dois anos)

In [ ]:
d10 = run(con, Q, "confianca_ia")
d10["ano"] = d10["year"].astype(str)
ordem_conf = ["Highly distrust", "Somewhat distrust", "Neither trust nor distrust",
              "Somewhat trust", "Highly trust"]
fig = px.bar(d10, y="ano", x="pct", color="ai_accuracy", orientation="h", barmode="stack",
             category_orders={"ai_accuracy": ordem_conf},
             color_discrete_sequence=["#c0392b", "#e6a199", "#cccccc", "#9fc5e8", "#2e75b6"],
             title="Confiança na precisão da IA (apenas quem usa IA)",
             labels={"pct": "% dos utilizadores", "ano": "", "ai_accuracy": ""})
fig.show(); save_fig(fig, "g10_confianca_ia")
# resumo: desconfia (as duas categorias vermelhas) vs confia
resumo = (d10.assign(grupo=d10["ai_accuracy"].map({
              "Highly distrust": "desconfia", "Somewhat distrust": "desconfia",
              "Neither trust nor distrust": "neutro",
              "Somewhat trust": "confia", "Highly trust": "confia"}))
          .groupby(["year", "grupo"])["pct"].sum().unstack().round(1))
resumo

## Conclusões do Dia 3 (preencher)

**Q1, quem usa IA:**
-

**Q2, salário e IA:**
-

**Q3, tecnologias:**
-

**Q4, confiança:**
-

**Limitações:**
- Descritivo, não causal; a composição da amostra mudou entre anos (experiência, países).
- Salários em USD nominais, autodeclarados; só quem declarou salário entra no dataset limpo.
- Medianas com valores repetidos (muitos respondentes declaram o mesmo valor): rankings finos não são fiáveis.